# Ranking Metrics

**Interview Focus**: NDCG vs MAP, precision/recall at K, MRR, handling ties.

**Key Concepts**:
- Precision@K, Recall@K: basic set-based ranking metrics
- MAP (Mean Average Precision): average precision across recall levels
- NDCG (Normalized Discounted Cumulative Gain): graded relevance with position discount
- MRR (Mean Reciprocal Rank): finding the first relevant item

**Math Notes**:
- DCG@K = $\sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)}$ (position discount via log)
- NDCG@K = DCG@K / IDCG@K (normalized to [0,1])
- AP = $\frac{1}{|R|} \sum_{k=1}^{n} P(k) \cdot \text{rel}(k)$ (average of precision at each relevant hit)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Toy Ranking Data

We simulate a search/recommendation system. For each query, we have a ranked list of items and their relevance labels.

In [ ]:
# Binary relevance: 1 = relevant, 0 = not relevant
# Each list represents a ranked result for one query (position 0 = rank 1)

# Good ranking: relevant items near the top
ranking_good = np.array([1, 1, 1, 0, 1, 0, 0, 0, 1, 0])

# Mediocre ranking: relevant items scattered
ranking_med  = np.array([0, 1, 0, 0, 1, 1, 0, 1, 0, 1])

# Bad ranking: relevant items at the bottom
ranking_bad  = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# Graded relevance (for NDCG): 0=irrelevant, 1=somewhat, 2=relevant, 3=highly relevant
graded_good = np.array([3, 3, 2, 0, 1, 0, 0, 0, 1, 0])
graded_med  = np.array([0, 2, 0, 0, 1, 3, 0, 1, 0, 2])
graded_bad  = np.array([0, 0, 0, 0, 0, 1, 2, 3, 3, 2])

print("Good ranking (binary):", ranking_good)
print("Good ranking (graded):", graded_good)

## 2. Precision@K and Recall@K

- **Precision@K**: fraction of top-K items that are relevant
- **Recall@K**: fraction of all relevant items found in top-K

In [ ]:
def precision_at_k(relevance, k):
    """Fraction of top-k results that are relevant."""
    assert k >= 1 and k <= len(relevance)
    return np.sum(relevance[:k] > 0) / k


def recall_at_k(relevance, k):
    """Fraction of all relevant items found in top-k."""
    total_relevant = np.sum(relevance > 0)
    if total_relevant == 0:
        return 0.0
    return np.sum(relevance[:k] > 0) / total_relevant


# Compute for all K values
K_values = range(1, len(ranking_good) + 1)

print(f"{'K':>3} {'P@K (good)':>12} {'P@K (med)':>12} {'P@K (bad)':>12}")
print("-" * 42)
for k in K_values:
    print(f"{k:>3} {precision_at_k(ranking_good, k):>12.3f} "
          f"{precision_at_k(ranking_med, k):>12.3f} "
          f"{precision_at_k(ranking_bad, k):>12.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

K_values = list(range(1, len(ranking_good) + 1))

# Precision@K
ax = axes[0]
for ranking, label, color in [(ranking_good, 'Good', '#2196F3'),
                               (ranking_med, 'Mediocre', '#FF9800'),
                               (ranking_bad, 'Bad', '#F44336')]:
    p_at_k = [precision_at_k(ranking, k) for k in K_values]
    ax.plot(K_values, p_at_k, 'o-', color=color, label=label, linewidth=2)
ax.set_xlabel('K')
ax.set_ylabel('Precision@K')
ax.set_title('Precision@K')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(K_values)

# Recall@K
ax = axes[1]
for ranking, label, color in [(ranking_good, 'Good', '#2196F3'),
                               (ranking_med, 'Mediocre', '#FF9800'),
                               (ranking_bad, 'Bad', '#F44336')]:
    r_at_k = [recall_at_k(ranking, k) for k in K_values]
    ax.plot(K_values, r_at_k, 'o-', color=color, label=label, linewidth=2)
ax.set_xlabel('K')
ax.set_ylabel('Recall@K')
ax.set_title('Recall@K')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(K_values)

plt.tight_layout()
plt.show()

print("Note: Recall@K converges to 1.0 for all — every ranking eventually finds all items.")
print("Precision@K and Recall@K converge to the same value at K=N.")

## 3. Average Precision (AP) and MAP

Average Precision summarizes the precision-recall curve into a single number.

$$AP = \frac{1}{|\text{relevant}|} \sum_{k=1}^{n} P@k \cdot \text{rel}(k)$$

Only positions where a relevant item appears contribute. MAP = mean AP across queries.

In [ ]:
def average_precision(relevance):
    """Compute Average Precision for a single ranked list (binary relevance)."""
    relevance = np.array(relevance, dtype=float)
    n_relevant = np.sum(relevance > 0)
    if n_relevant == 0:
        return 0.0
    
    ap = 0.0
    hits = 0
    for k in range(len(relevance)):
        if relevance[k] > 0:
            hits += 1
            ap += hits / (k + 1)  # P@(k+1) at this relevant position
    
    return ap / n_relevant


def mean_average_precision(rankings):
    """MAP: average AP across multiple queries."""
    return np.mean([average_precision(r) for r in rankings])


# Detailed walkthrough for the good ranking
print("AP walkthrough for good ranking:", ranking_good)
print(f"{'Position':>10} {'Relevant':>10} {'Hits':>6} {'P@k':>8} {'Contribution':>14}")
print("-" * 52)
hits = 0
total_contribution = 0.0
for k in range(len(ranking_good)):
    if ranking_good[k] > 0:
        hits += 1
        p_at_k = hits / (k + 1)
        total_contribution += p_at_k
        print(f"{k+1:>10} {'Yes':>10} {hits:>6} {p_at_k:>8.3f} {p_at_k:>14.3f}")
    else:
        print(f"{k+1:>10} {'No':>10} {hits:>6} {hits/(k+1):>8.3f} {'—':>14}")

n_rel = np.sum(ranking_good > 0)
print(f"\nAP = {total_contribution:.3f} / {n_rel} = {total_contribution/n_rel:.4f}")

# Verify
print(f"\nAP (good):     {average_precision(ranking_good):.4f}")
print(f"AP (mediocre): {average_precision(ranking_med):.4f}")
print(f"AP (bad):      {average_precision(ranking_bad):.4f}")

In [ ]:
# MAP across multiple queries
np.random.seed(42)
n_queries = 20
n_items = 10
n_relevant_per_query = 4

# Generate random rankings of varying quality
good_rankings = []
random_rankings = []
for _ in range(n_queries):
    # Good system: relevant items tend to be ranked higher
    rel = np.zeros(n_items)
    positions = np.random.choice(n_items, n_relevant_per_query, replace=False)
    positions_good = np.sort(np.random.choice(n_items, n_relevant_per_query, replace=False))  # bias toward top
    rel_good = np.zeros(n_items)
    # Bias relevant items toward top positions
    top_positions = np.random.choice(range(min(6, n_items)), min(3, n_relevant_per_query), replace=False)
    remaining = np.random.choice([i for i in range(n_items) if i not in top_positions],
                                  n_relevant_per_query - len(top_positions), replace=False)
    all_pos = np.concatenate([top_positions, remaining])
    rel_good[all_pos.astype(int)] = 1
    good_rankings.append(rel_good)
    
    # Random system
    rel_rand = np.zeros(n_items)
    rand_positions = np.random.choice(n_items, n_relevant_per_query, replace=False)
    rel_rand[rand_positions] = 1
    random_rankings.append(rel_rand)

map_good = mean_average_precision(good_rankings)
map_random = mean_average_precision(random_rankings)

print(f"MAP (good system):   {map_good:.4f}")
print(f"MAP (random system): {map_random:.4f}")

## 4. DCG and NDCG

Unlike MAP, NDCG handles **graded relevance** (not just binary).

**DCG@K** = $\sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)}$

- Numerator: exponential gain from relevance (highly relevant items contribute much more)
- Denominator: logarithmic discount by position (later positions count less)

**NDCG@K** = DCG@K / IDCG@K where IDCG is the DCG of the ideal ranking (sorted by relevance desc).

In [ ]:
def dcg_at_k(relevance, k):
    """Discounted Cumulative Gain at position K."""
    relevance = np.array(relevance, dtype=float)[:k]
    positions = np.arange(1, len(relevance) + 1)
    # Gain: 2^rel - 1, Discount: log2(position + 1)
    gains = (2 ** relevance - 1) / np.log2(positions + 1)
    return np.sum(gains)


def ndcg_at_k(relevance, k):
    """Normalized DCG: DCG / ideal DCG."""
    dcg = dcg_at_k(relevance, k)
    # Ideal: sort relevance descending
    ideal_relevance = np.sort(relevance)[::-1]
    idcg = dcg_at_k(ideal_relevance, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg


# Detailed walkthrough
print("DCG walkthrough for graded_good:", graded_good)
print(f"{'Pos':>4} {'Rel':>5} {'Gain (2^rel-1)':>16} {'Discount':>10} {'Contribution':>14}")
print("-" * 52)
dcg_total = 0.0
for i in range(len(graded_good)):
    gain = 2 ** graded_good[i] - 1
    discount = np.log2(i + 2)  # log2(position + 1), position is 1-indexed
    contribution = gain / discount
    dcg_total += contribution
    print(f"{i+1:>4} {graded_good[i]:>5} {gain:>16.1f} {discount:>10.3f} {contribution:>14.4f}")

print(f"\nDCG@10 = {dcg_total:.4f}")
print(f"Verify: {dcg_at_k(graded_good, 10):.4f}")

In [ ]:
# Compare NDCG across ranking qualities
print("NDCG comparison (graded relevance):")
print(f"{'K':>3} {'NDCG (good)':>14} {'NDCG (med)':>14} {'NDCG (bad)':>14}")
print("-" * 48)
for k in [1, 3, 5, 10]:
    print(f"{k:>3} {ndcg_at_k(graded_good, k):>14.4f} "
          f"{ndcg_at_k(graded_med, k):>14.4f} "
          f"{ndcg_at_k(graded_bad, k):>14.4f}")

print("\nNDCG@10 is always 1.0 if all items are present (just reordered).")
print("The power of NDCG is at smaller K — it measures if the BEST items are at the TOP.")

In [ ]:
# Visualize the discount function
positions = np.arange(1, 21)
discounts = 1.0 / np.log2(positions + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Discount curve
ax = axes[0]
ax.bar(positions, discounts, color='#2196F3', alpha=0.7, edgecolor='white')
ax.set_xlabel('Position')
ax.set_ylabel('Discount Factor: 1 / log2(pos+1)')
ax.set_title('Position Discount in DCG')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(positions)

# NDCG@K for all three rankings
ax = axes[1]
K_range = range(1, 11)
for graded, label, color in [(graded_good, 'Good', '#2196F3'),
                              (graded_med, 'Mediocre', '#FF9800'),
                              (graded_bad, 'Bad', '#F44336')]:
    ndcg_vals = [ndcg_at_k(graded, k) for k in K_range]
    ax.plot(list(K_range), ndcg_vals, 'o-', color=color, label=label, linewidth=2)
ax.set_xlabel('K')
ax.set_ylabel('NDCG@K')
ax.set_title('NDCG@K Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(list(K_range))

plt.tight_layout()
plt.show()

## 5. Mean Reciprocal Rank (MRR)

MRR focuses on **where the first relevant item appears**.

$$RR = \frac{1}{\text{rank of first relevant item}}$$

$$MRR = \frac{1}{|Q|} \sum_{q=1}^{|Q|} RR_q$$

Common in: QA systems, navigational search (user wants one specific answer).

In [ ]:
def reciprocal_rank(relevance):
    """1 / rank of first relevant item. 0 if none relevant."""
    relevance = np.array(relevance)
    relevant_positions = np.where(relevance > 0)[0]
    if len(relevant_positions) == 0:
        return 0.0
    return 1.0 / (relevant_positions[0] + 1)  # +1 for 1-indexed rank


def mean_reciprocal_rank(rankings):
    """MRR across multiple queries."""
    return np.mean([reciprocal_rank(r) for r in rankings])


# Demo
print(f"RR (good ranking):     {reciprocal_rank(ranking_good):.4f}  (first relevant at position 1)")
print(f"RR (mediocre ranking): {reciprocal_rank(ranking_med):.4f}  (first relevant at position 2)")
print(f"RR (bad ranking):      {reciprocal_rank(ranking_bad):.4f}  (first relevant at position 6)")

# MRR with multiple queries
query_results = [
    [0, 0, 1, 0, 1],  # first relevant at position 3 -> RR = 1/3
    [1, 0, 0, 0, 0],  # first relevant at position 1 -> RR = 1
    [0, 1, 0, 1, 0],  # first relevant at position 2 -> RR = 1/2
    [0, 0, 0, 0, 1],  # first relevant at position 5 -> RR = 1/5
    [0, 0, 0, 0, 0],  # no relevant -> RR = 0
]

print(f"\nMRR across 5 queries: {mean_reciprocal_rank(query_results):.4f}")
print("Individual RRs:", [f"{reciprocal_rank(r):.3f}" for r in query_results])

## 6. Full Evaluation: Toy Recommendation System

In [ ]:
def evaluate_ranking(relevance_lists, k=5):
    """Comprehensive ranking evaluation."""
    results = {
        'P@K': np.mean([precision_at_k(r, k) for r in relevance_lists]),
        'R@K': np.mean([recall_at_k(r, k) for r in relevance_lists]),
        'MAP': mean_average_precision(relevance_lists),
        'MRR': mean_reciprocal_rank(relevance_lists),
    }
    # For NDCG, use the same relevance (treating binary as graded)
    results['NDCG@K'] = np.mean([ndcg_at_k(r, k) for r in relevance_lists])
    return results


# Simulate three recommendation systems with 50 queries, 20 items each
np.random.seed(42)
n_queries = 50
n_items = 20
n_relevant = 5

def make_system_rankings(quality='good'):
    """Generate rankings of varying quality."""
    rankings = []
    for _ in range(n_queries):
        rel = np.zeros(n_items)
        if quality == 'good':
            # Relevant items in top positions
            positions = np.sort(np.random.choice(min(8, n_items), n_relevant, replace=False))
        elif quality == 'mediocre':
            # Relevant items scattered
            positions = np.random.choice(n_items, n_relevant, replace=False)
        else:  # bad
            # Relevant items at bottom
            positions = np.sort(np.random.choice(range(n_items//2, n_items), n_relevant, replace=False))
        rel[positions] = 1
        rankings.append(rel)
    return rankings

systems = {
    'System A (good)': make_system_rankings('good'),
    'System B (mediocre)': make_system_rankings('mediocre'),
    'System C (bad)': make_system_rankings('bad'),
}

K = 5
print(f"Evaluation at K={K} across {n_queries} queries:")
print(f"\n{'System':<25} {'P@5':>8} {'R@5':>8} {'MAP':>8} {'NDCG@5':>8} {'MRR':>8}")
print("-" * 70)
for name, rankings in systems.items():
    results = evaluate_ranking(rankings, k=K)
    print(f"{name:<25} {results['P@K']:>8.4f} {results['R@K']:>8.4f} "
          f"{results['MAP']:>8.4f} {results['NDCG@K']:>8.4f} {results['MRR']:>8.4f}")

In [ ]:
# Visualize metric comparison
metric_names = ['P@5', 'R@5', 'MAP', 'NDCG@5', 'MRR']
system_names = list(systems.keys())
colors = ['#2196F3', '#FF9800', '#F44336']

all_results = {}
for name, rankings in systems.items():
    r = evaluate_ranking(rankings, k=K)
    all_results[name] = [r['P@K'], r['R@K'], r['MAP'], r['NDCG@K'], r['MRR']]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metric_names))
width = 0.25

for i, (name, vals) in enumerate(all_results.items()):
    ax.bar(x + i * width, vals, width, label=name, color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Ranking Metrics Comparison Across Systems')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 7. Handling Ties

When items have the same predicted score, ranking order is ambiguous. Common strategies:

1. **Average**: average the metric over all possible orderings of tied items
2. **Optimistic**: assume best ordering for tied items
3. **Pessimistic**: assume worst ordering
4. **Random**: break ties randomly (most common in practice)

In [ ]:
def rank_with_ties(scores, relevance, strategy='random', seed=42):
    """Produce a ranking from scores, handling ties with different strategies."""
    scores = np.array(scores, dtype=float)
    relevance = np.array(relevance, dtype=float)
    
    if strategy == 'optimistic':
        # Within tied groups, put relevant items first
        order = np.lexsort((-relevance, -scores))
    elif strategy == 'pessimistic':
        # Within tied groups, put relevant items last
        order = np.lexsort((relevance, -scores))
    elif strategy == 'random':
        rng = np.random.RandomState(seed)
        noise = rng.uniform(0, 1e-10, len(scores))
        order = np.argsort(-(scores + noise))
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    return relevance[order]


# Demo: tied scores
scores = [0.9, 0.7, 0.7, 0.7, 0.3, 0.3, 0.1, 0.1, 0.1, 0.1]
relevance = [1, 0, 1, 0, 1, 0, 1, 0, 0, 1]

print("Scores:   ", scores)
print("Relevance:", relevance)
print("\nNote: items at positions 2-4 are tied (score=0.7), 2 of 3 are relevant.")
print()

for strategy in ['optimistic', 'pessimistic', 'random']:
    ranked_rel = rank_with_ties(scores, relevance, strategy)
    ap = average_precision(ranked_rel)
    ndcg5 = ndcg_at_k(ranked_rel, 5)
    print(f"{strategy:>12}: ranked_rel = {ranked_rel.astype(int).tolist()}, AP = {ap:.4f}, NDCG@5 = {ndcg5:.4f}")

print("\nTies can significantly affect metrics — always document your tie-breaking strategy.")

## 8. NDCG with Different Gain Functions

In [ ]:
def dcg_at_k_linear(relevance, k):
    """DCG with linear gain: rel_i / log2(i+1) instead of (2^rel_i - 1) / log2(i+1)."""
    relevance = np.array(relevance, dtype=float)[:k]
    positions = np.arange(1, len(relevance) + 1)
    return np.sum(relevance / np.log2(positions + 1))


def ndcg_at_k_linear(relevance, k):
    """NDCG with linear gain function."""
    dcg = dcg_at_k_linear(relevance, k)
    ideal = np.sort(relevance)[::-1]
    idcg = dcg_at_k_linear(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg


# Compare gain functions
print("Relevance levels and their gains:")
print(f"{'Relevance':>10} {'Linear gain':>14} {'Exponential gain':>18}")
print("-" * 45)
for rel in [0, 1, 2, 3, 4, 5]:
    print(f"{rel:>10} {rel:>14} {2**rel - 1:>18}")

print("\nExponential gain emphasizes the difference between 'good' and 'great' relevance.")
print("Example: rel=3 vs rel=2 is 7 vs 3 with exp gain, but only 3 vs 2 with linear.")

# Compare on our graded rankings
print(f"\nNDCG@5 comparison:")
print(f"{'Ranking':<12} {'Exponential':>14} {'Linear':>14}")
print("-" * 42)
for name, graded in [('Good', graded_good), ('Mediocre', graded_med), ('Bad', graded_bad)]:
    print(f"{name:<12} {ndcg_at_k(graded, 5):>14.4f} {ndcg_at_k_linear(graded, 5):>14.4f}")

## 9. Metric Sensitivity Analysis

In [ ]:
# How does swapping two items affect each metric?
base_ranking = np.array([1, 1, 0, 0, 1, 0, 0, 1, 0, 0])  # base: 4 relevant

print("Base ranking:", base_ranking.tolist())
print(f"Base AP={average_precision(base_ranking):.4f}, NDCG@5={ndcg_at_k(base_ranking, 5):.4f}, "
      f"MRR={reciprocal_rank(base_ranking):.4f}, P@5={precision_at_k(base_ranking, 5):.4f}")
print()

# Try swapping different pairs
swaps = [(0, 2), (0, 9), (4, 5), (2, 7)]  # (pos_i, pos_j) to swap

print(f"{'Swap':>12} {'New Ranking':<30} {'AP':>8} {'NDCG@5':>8} {'MRR':>6} {'P@5':>6}")
print("-" * 75)
for i, j in swaps:
    swapped = base_ranking.copy()
    swapped[i], swapped[j] = swapped[j], swapped[i]
    print(f"  ({i},{j})      {str(swapped.tolist()):<30} "
          f"{average_precision(swapped):>8.4f} {ndcg_at_k(swapped, 5):>8.4f} "
          f"{reciprocal_rank(swapped):>6.3f} {precision_at_k(swapped, 5):>6.3f}")

print("\nNote: Swapping a top relevant item to the bottom hurts AP/NDCG the most.")
print("MRR only cares about the first relevant item position.")

In [ ]:
# Visualize: how each metric changes as we progressively degrade a perfect ranking
n_items = 10
n_relevant = 5

# Start with perfect ranking
perfect = np.array([1]*n_relevant + [0]*(n_items - n_relevant))

# Progressively move one relevant item to a worse position
degradation_steps = []
metrics_over_degradation = {'AP': [], 'NDCG@5': [], 'MRR': [], 'P@3': []}

for step in range(n_items):
    # Take the perfect ranking and move the first relevant item to position 'step'
    ranking = perfect.copy()
    if step > 0:
        # Randomly swap step positions
        np.random.seed(step)
        for _ in range(step):
            i = np.random.randint(0, n_items)
            j = np.random.randint(0, n_items)
            ranking[i], ranking[j] = ranking[j], ranking[i]
    
    degradation_steps.append(step)
    metrics_over_degradation['AP'].append(average_precision(ranking))
    metrics_over_degradation['NDCG@5'].append(ndcg_at_k(ranking, 5))
    metrics_over_degradation['MRR'].append(reciprocal_rank(ranking))
    metrics_over_degradation['P@3'].append(precision_at_k(ranking, 3))

fig, ax = plt.subplots(figsize=(9, 5))
for metric, vals in metrics_over_degradation.items():
    ax.plot(degradation_steps, vals, 'o-', label=metric, linewidth=2)
ax.set_xlabel('Number of Random Swaps (degradation)')
ax.set_ylabel('Metric Value')
ax.set_title('Metric Sensitivity to Ranking Degradation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Interview Questions & Answers

---

**Q: NDCG vs MAP?**

- **MAP** uses **binary** relevance only (relevant or not). It computes average precision at each relevant hit, then averages across queries.
- **NDCG** handles **graded** relevance (e.g., 0/1/2/3 stars). The exponential gain 2^rel - 1 heavily rewards placing highly relevant items at the top.
- Use MAP when relevance is binary (web search: clicked or not). Use NDCG when you have graded labels (explicit ratings, multi-level annotations).
- NDCG is normalized to [0,1]. MAP is also in [0,1] but with different semantics.

---

**Q: What does 'discounted' mean in DCG?**

The log2(position + 1) denominator acts as a **position discount** — items ranked lower receive less credit. This models the real-world intuition that users are less likely to examine results further down the list. The logarithmic discount is gentle: position 10 still gets ~30% of the credit of position 1.

---

**Q: How to handle ties?**

When items share the same score, their relative order is undefined. Options:
1. **Random tie-breaking** (most common): adds noise to break ties, report average over multiple random seeds
2. **Optimistic/Pessimistic**: bound the metric by assuming best/worst case ordering of tied items
3. **Expected metric**: analytically compute the expected value averaging over all possible orderings of tied items

Always document your tie-breaking strategy for reproducibility.

---

**Q: When to use MRR vs MAP vs NDCG?**

- **MRR**: user needs ONE answer (QA, navigational search, "I'm feeling lucky"). Only the first relevant result matters.
- **MAP**: user wants MANY relevant results with binary relevance (document retrieval, web search).
- **NDCG**: results have graded relevance (ratings, multi-level annotations). Standard for learning-to-rank benchmarks.
- **P@K/R@K**: when you care about a fixed cutoff (show top-10 recommendations).

## Quick Reference

```
Precision@K = |relevant in top K| / K
Recall@K    = |relevant in top K| / |total relevant|
AP          = (1/|R|) * sum of P@k at each relevant position
MAP         = mean AP across queries
DCG@K       = sum_i (2^rel_i - 1) / log2(i+1)
NDCG@K      = DCG@K / IDCG@K
RR          = 1 / rank_of_first_relevant
MRR         = mean RR across queries
```

| Metric | Relevance | Use Case | Range |
|--------|-----------|----------|-------|
| P@K    | Binary    | Fixed cutoff | [0,1] |
| MAP    | Binary    | Full ranking | [0,1] |
| NDCG   | Graded    | Full ranking | [0,1] |
| MRR    | Binary    | First hit    | [0,1] |